# Gradient Sports API — Exemplos de Uso

Importa o cliente de `gradient_client.py`.  
A autenticação é lida automaticamente do arquivo `.env` (`BEARER_TOKEN`).

In [2]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent.parent))
from src.gradient_client import GradientSportsClient

pd.set_option('display.max_columns', None)

### Status da requisição

In [3]:
client = GradientSportsClient()

# health check
client.get_status()

{'data': {'status': 'ok'}}

### Competições

In [4]:
# competitions & seasons the account can access
df_competitions = client.get_competitions(as_dataframe=True).drop('datasets', axis=1).drop_duplicates().reset_index(drop=True)
df_competitions

,season,competition.id,competition.name
0,2020-2021,1,Premier League
1,2021-2022,1,Premier League
2,2022-2023,1,Premier League
3,2023-2024,1,Premier League
4,2024-2025,1,Premier League
5,2025-2026,1,Premier League
6,2023,42,Brasileiro Série A
7,2024,42,Brasileiro Série A
8,2025,42,Brasileiro Série A
9,2026,42,Brasileiro Série A


- Existem dados de duas competições, Premier League e Brasileiro Série A.
- Dentre essas competições, existem dados de 6 temporadas da Premier League e 4 temporadas do Brasileiro Série A.

In [5]:
# teams the account can access
df_teams = client.get_teams(as_dataframe=True).drop('dataset', axis=1).drop_duplicates().reset_index(drop=True)
df_teams.sort_values('competition.id')

,team.id,team.name,competition.id,competition.name
0,1.0,AFC Bournemouth,1,Premier League
1,20.0,Wolverhampton Wanderers,1,Premier League
6,221.0,Nottingham Forest,1,Premier League
7,335.0,Sunderland AFC,1,Premier League
12,10.0,Liverpool,1,Premier League
13,218.0,Luton Town,1,Premier League
11,7.0,Crystal Palace,1,Premier League
8,16.0,Southampton,1,Premier League
24,55.0,Leeds United,1,Premier League
25,13.0,Newcastle United,1,Premier League


In [6]:
df_teams.info()

<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   team.id           57 non-null     float64
 1   team.name         57 non-null     str    
 2   competition.id    59 non-null     int64  
 3   competition.name  59 non-null     str    
dtypes: float64(1), int64(1), str(2)
memory usage: 2.0 KB


In [7]:
df_teams.duplicated().sum()

np.int64(0)

- O dataset de times possui duas colunas com ids dos times nulas e não possui nenhuma duplicata. As colunas com ids dos times nulas devem ser removidas.

In [8]:
df_teams = df_teams[df_teams['team.name'].notna()].reset_index()

In [9]:
df_teams['competition.name'].value_counts()

competition.name
Brasileiro Série A    29
Premier League        28
Name: count, dtype: int64

- Entre as temporadas, existem dados de 28 times da Premier League e 29 do Brasileirão.

### Jogos

In [10]:
# all games — or filter by season / competition / team
# as_dataframe=True → one row per game, nested dicts dot-expanded
games = client.get_games(as_dataframe=True)
games

,id,date,season,teamExtraTimeStartSide,teamStartSide,venueType,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,stadium.name,stadium.length,stadium.width
0,32019,2024-08-31,2024-2025,Left,Right,OPPONENT_HOME,3,Aston Villa,1,Premier League,9,Leicester City,King Power Stadium,105.0,68.0
1,4646,2023-02-04,2022-2023,Right,Right,TEAM_HOME,3,Aston Villa,1,Premier League,9,Leicester City,Villa Park,105.0,68.0
2,259,2020-12-07,2020-2021,Right,Left,TEAM_HOME,4,Brighton & Hove Albion,1,Premier League,16,Southampton,American Express Stadium,105.0,68.0
3,40871,2025-08-17,2025-2026,Left,Right,OPPONENT_HOME,119,Brentford,1,Premier League,221,Nottingham Forest,The City Ground,102.5,68.0
4,32192,2025-01-04,2024-2025,Left,Right,OPPONENT_HOME,119,Brentford,1,Premier League,16,Southampton,St. Mary's Stadium,105.0,68.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3407,13556,2024-02-04,2023-2024,Right,Left,TEAM_HOME,1,AFC Bournemouth,1,Premier League,221,Nottingham Forest,Vitality Stadium,105.0,68.0
3408,4725,2023-04-08,2022-2023,Right,Right,TEAM_HOME,3,Aston Villa,1,Premier League,221,Nottingham Forest,Villa Park,105.0,68.0
3409,186,2020-10-04,2020-2021,Right,Left,TEAM_HOME,12,Manchester United,1,Premier League,17,Tottenham Hotspur,Old Trafford,105.0,68.0
3410,37019,2025-11-15,2025,Left,Left,OPPONENT_HOME,435,Flamengo,42,Brasileiro Série A,873,Sport Recife,Estádio Ilha do Retiro,105.0,68.0


In [ ]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 3412 entries, 0 to 3411
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      3412 non-null   int64  
 1   date                    3412 non-null   str    
 2   season                  3412 non-null   str    
 3   teamExtraTimeStartSide  3412 non-null   str    
 4   teamStartSide           3412 non-null   str    
 5   venueType               3412 non-null   str    
 6   team.id                 3412 non-null   int64  
 7   team.name               3412 non-null   str    
 8   competition.id          3412 non-null   int64  
 9   competition.name        3412 non-null   str    
 10  opponentTeam.id         3412 non-null   int64  
 11  opponentTeam.name       3412 non-null   str    
 12  stadium.name            3412 non-null   str    
 13  stadium.length          3412 non-null   float64
 14  stadium.width           3412 non-null   float64
dty

In [12]:
games.duplicated().sum()

np.int64(0)

- O conjunto de dados de jogos não parece ter linhas duplicadas ou dados ausentes.

### Eventos

In [13]:
game_id = games["id"].iloc[1]
game_season = games["season"].iloc[1]
game_competition = games['competition.id'].iloc[1]

# structured events — one row per possession event
df_game_events = client.get_game_events(game_id, as_dataframe=True)

df_game_events['competitionId'] = game_competition
df_game_events['gameId'] = game_id
df_game_events['season'] = game_season

df_game_events.head()

,id,competitionId,gameId,season,period,periodDescription,startGameClock,startFormattedGameClock,homeTeam,gameEventType,gameEventTypeDescription,setpieceType,setpieceTypeDescription,touches,touchesInBox,videoMissingpossessionEvents,homePlayers,awayPlayers,balls,team.id,team.name,player.id,player.name
0,7488643,1,4646,2022-2023,1,First half,0,00:00,False,FIRSTKICKOFF,First half kick off,K,Kickoff,1.0,0,None,"[{'speed': None, 'y': 9.035, 'x': 15.688, 'pla...","[{'speed': None, 'y': 9.381, 'x': -0.203, 'pla...","[{'z': 0, 'y': -0.19, 'x': -0.69, 'visibility'...",9.0,Leicester City,7812.0,Mateus Tetê
1,7488646,1,4646,2022-2023,1,First half,0,00:00,False,OTB,A possession with a player on the ball,O,Open Play,2.0,0,None,"[{'speed': None, 'y': 11.171, 'x': 17.939, 'pl...","[{'speed': None, 'y': 6.76, 'x': 0.669, 'playe...","[{'z': 0.92, 'y': 0.57, 'x': -5.65, 'visibilit...",9.0,Leicester City,215.0,Youri Tielemans
2,7488651,1,4646,2022-2023,1,First half,4,00:04,True,OTB,A possession with a player on the ball,O,Open Play,1.0,0,None,"[{'speed': None, 'y': 17.136, 'x': 22.427, 'pl...","[{'speed': None, 'y': 4.927, 'x': 7.746, 'play...","[{'z': 3.42, 'y': 21.77, 'x': 17.73, 'visibili...",3.0,Aston Villa,2062.0,Jacob Ramsey
3,7488655,1,4646,2022-2023,1,First half,6,00:06,False,OTB,A possession with a player on the ball,O,Open Play,3.0,0,None,"[{'speed': None, 'y': 21.354, 'x': 17.289, 'pl...","[{'speed': None, 'y': 7.968, 'x': 9.752, 'play...","[{'z': 0.06, 'y': 24.34, 'x': 9.57, 'visibilit...",9.0,Leicester City,15632.0,Victor Kristiansen
4,7488662,1,4646,2022-2023,1,First half,8,00:08,False,OTB,A possession with a player on the ball,O,Open Play,3.0,0,None,"[{'speed': None, 'y': 23.713, 'x': 12.251, 'pl...","[{'speed': None, 'y': 8.878, 'x': 8.583, 'play...","[{'z': 0.49, 'y': 26.11, 'x': 12.72, 'visibili...",9.0,Leicester City,220.0,James Maddison


In [15]:
# pick a game id from the list above
game_id = games["id"].iloc[1]
game_season = games["season"].iloc[1]
game_competition = games['competition.id'].iloc[1]

# structured events — one row per possession event
df_game_events_flat = client.get_game_events_flat(game_id, as_dataframe=True)

df_game_events_flat['competitionId'] = game_competition
df_game_events_flat['gameId'] = game_id
df_game_events_flat['season'] = game_season

df_game_events_flat.head()

,id,competitionId,gameId,season,period,periodDescription,eventType,eventTypeDescription,startGameClock,startFormattedGameClock,homeTeam,details,homePlayers,awayPlayers,balls,player.id,player.name,team.id,team.name
0,0f0a3dfaab810758f037fc4e2d8771d9,1,4646,2022-2023,1,First half,FIRSTKICKOFF,First half kick off,0,00:00,False,"{'earlyDistribution': False, 'endType': None, ...","[{'speed': None, 'y': 9.035, 'x': 15.688, 'pla...","[{'speed': None, 'y': 9.381, 'x': -0.203, 'pla...","[{'z': 0, 'y': -0.19, 'x': -0.69, 'visibility'...",7812.0,Mateus Tetê,9.0,Leicester City
1,bc54b4833301b67fde64a5bff5a0da83,1,4646,2022-2023,1,First half,PA,Pass,0,00:00,False,"{'secondIncompletionReasonType': None, 'endGam...","[{'speed': None, 'y': 9.035, 'x': 15.688, 'pla...","[{'speed': None, 'y': 9.381, 'x': -0.203, 'pla...","[{'z': 0, 'y': -0.19, 'x': -0.69, 'visibility'...",7812.0,Mateus Tetê,9.0,Leicester City
2,d61ab5c1d3bc8d80e35c7432e86134ad,1,4646,2022-2023,1,First half,PA,Pass,0,00:00,False,"{'secondIncompletionReasonType': 'UH', 'endGam...","[{'speed': None, 'y': 11.171, 'x': 17.939, 'pl...","[{'speed': None, 'y': 6.76, 'x': 0.669, 'playe...","[{'z': 0.92, 'y': 0.57, 'x': -5.65, 'visibilit...",215.0,Youri Tielemans,9.0,Leicester City
3,b67a259568ea0ed355cec8dd7d1e137a,1,4646,2022-2023,1,First half,OTB,A possession with a player on the ball,0,00:00,False,"{'earlyDistribution': False, 'endType': None, ...","[{'speed': None, 'y': 11.171, 'x': 17.939, 'pl...","[{'speed': None, 'y': 6.76, 'x': 0.669, 'playe...","[{'z': 0.92, 'y': 0.57, 'x': -5.65, 'visibilit...",215.0,Youri Tielemans,9.0,Leicester City
4,45c31bc7bce2bce7773c49a8f509655c,1,4646,2022-2023,1,First half,OTB,A possession with a player on the ball,4,00:04,True,"{'earlyDistribution': False, 'endType': None, ...","[{'speed': None, 'y': 17.136, 'x': 22.427, 'pl...","[{'speed': None, 'y': 4.927, 'x': 7.746, 'play...","[{'z': 3.42, 'y': 21.77, 'x': 17.73, 'visibili...",2062.0,Jacob Ramsey,3.0,Aston Villa
